# Ensemble of Specialized Mixture of Experts (MoE) for NLI
This notebook implements a state-of-the-art Natural Language Inference pipeline combining:
1. **T5 Data Augmentation**: Generating synthetic hypotheses to improve robustness.
2. **POS-Specialized MoE**: A custom architecture with experts for Semantics, Entities, Actions, and Logic.
3. **Model Ensembling**: Averaging predictions from **DeBERTa-v3** and **ModernBERT** backbones.

In [12]:
!pip install datasets transformers torch spacy pandas numpy scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 28.5 MB/s eta 0:00:00m eta 0:00:010:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [13]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import spacy
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score, accuracy_score
import os
import torch
import numpy as np
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, DefaultDataCollator, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset

nlp = spacy.load("en_core_web_sm", disable=["ner"])
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Data Augmentation (T5)
Use a generative transformer model to expand the available dataset by paraphrasing the existing examples.

In [14]:
def augment_dataset(df):
    # Model responsible for paraphrasing the existing examples
    generativeModel = "google/flan-t5-base"
    t5Tokenizer = T5Tokenizer.from_pretrained(generativeModel)
    generator = T5ForConditionalGeneration.from_pretrained(generativeModel).to(device)

    # Selects a random sample of the existing training set
    sampleSize = 1000
    generator.eval()
    generatedExamples = []
    subset = df.sample(n=sampleSize)

    # For each of the sampled examples, generate a new hypothesis and premise
    for _, ex in subset.iterrows():
        promptHyp = f"make a sentence that means this: {ex['hypothesis']}"
        inputsHyp = t5Tokenizer(promptHyp, return_tensors="pt").to(device)
        outputsHyp = generator.generate(**inputsHyp, max_length=64)
        gen_hyp = t5Tokenizer.decode(outputsHyp[0], skip_special_tokens=True)

        promptPrem = f"make a sentence that means this: {ex['premise']}"
        inputsPrem = t5Tokenizer(promptPrem, return_tensors="pt").to(device)
        outputsPrem = generator.generate(**inputsPrem, max_length=64)
        gen_prem = t5Tokenizer.decode(outputsPrem[0], skip_special_tokens=True)

        generatedExamples.append({"premise": gen_prem, "hypothesis": gen_hyp, "label": ex["label"]})

    return pd.concat([df, pd.DataFrame(generatedExamples)]).reset_index(drop=True)

trainingDataframe = pd.read_csv("training_data/NLI/train.csv")
devDataframe = pd.read_csv("training_data/NLI/dev.csv")
trainingDataframeWithGenerated = augment_dataset(trainingDataframe)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## 2. Multi-Backbone MoE Architecture
This model is a Mixture of Experts (MoE) which has different "experts" that focus on different aspects of the input sentence.
The experts are then combined using a gating mechanism that decides how much to focus on each expert based on the input.
The experts are designed to focus on different parts of the sentence:
- The semantic expert focuses on the overall meaning of the sentence.
- The entity expert focuses on the entities mentioned in the sentence.
- The action expert focuses on the actions described in the sentence.
- The logic expert focuses on the logical structure of the sentence, using the logical features we extracted earlier.

In [15]:
class POSSpecializedMoE(nn.Module):
    def __init__(self, modelName, labelCount=2):
        # Load the baseline model
        super().__init__()
        self.encoder = AutoModel.from_pretrained(modelName).float()
        modelWidth = self.encoder.config.hidden_size
        num_logic_features = 4

        # Create experts which focus on different parts of the sentence
        self.semantic_expert = nn.Linear(modelWidth, modelWidth)
        self.entity_expert = nn.Linear(modelWidth, modelWidth)
        self.action_expert = nn.Linear(modelWidth, modelWidth)
        self.logic_expert = nn.Linear(num_logic_features, modelWidth)

        # Combines the results from the experts using a gating mechanism
        self.normalisationLayer = nn.LayerNorm(num_logic_features)
        self.gating = nn.Sequential(
            nn.Linear(modelWidth * 3 + num_logic_features, 128),
            nn.ReLU(),
            nn.Linear(128, 4),
            nn.Softmax(dim=-1)
        )

        self.classifier = nn.Linear(modelWidth, labelCount)
        self.lossFunc = nn.CrossEntropyLoss()

    # Encodes the input and extracts the [CLS] token representation
    def encode_cls(self, input_ids, attention_mask):
      outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
      cls_output = outputs.last_hidden_state[:, 0, :]
      return cls_output

    def forward(
        self,
        input_ids,
        attention_mask,
        entity_input_ids,
        entity_attention_mask,
        action_input_ids,
        action_attention_mask,
        logic_features=None,
        labels=None
    ):
        # Encode the main input, entity-focused input, and action-focused input
        main_cls = self.encode_cls(input_ids, attention_mask)
        entity_cls = self.encode_cls(entity_input_ids, entity_attention_mask)
        action_cls = self.encode_cls(action_input_ids, action_attention_mask)

        # If logic features are not provided, create a zero tensor
        if logic_features is None:
            logic_features = torch.zeros((input_ids.size(0), 4), device=input_ids.device)

        # Normalize logic features and pass through the logic expert
        logic_features = logic_features.float()
        norm_logic = self.normalisationLayer(logic_features)


        # Pass the CLS representations through their respective experts
        semanticRes = torch.tanh(self.semantic_expert(main_cls))
        entityRes = torch.tanh(self.entity_expert(entity_cls))
        actionRes = torch.tanh(self.action_expert(action_cls))
        logicRes = torch.tanh(self.logic_expert(norm_logic))


        # Use the gating mechanism to determine how much to rely on each expert
        gate_input = torch.cat([main_cls, entity_cls, action_cls, norm_logic], dim=-1)
        gate_weights = self.gating(gate_input)

        # Combine the expert outputs according to the gate weights
        experts = torch.stack([semanticRes, entityRes, actionRes, logicRes], dim=1)
        moe_output = torch.bmm(gate_weights.unsqueeze(1), experts).squeeze(1)
        
        logits = self.classifier(moe_output)

        loss = None
        if labels is not None:
            loss = self.lossFunc(logits, labels)

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

## 3. Preprocessing and Feature Extraction
We extract linguistic features using Spacy to feed the MoE gating and specialized experts.

In [16]:
# Extract words based on a specific POS
def get_pos_filtered_text(text, pos_tags):
    # Generate POS tags
    doc = nlp(str(text))
    # Only select those which have been specified
    tokens = [token.text for token in doc if token.pos_ in pos_tags]
    return " ".join(tokens) if tokens else "none"

def extract_logic_features(premise, hypothesis):
    # Produce POS tags for both the premise and hypothesis
    p_doc, h_doc = nlp(str(premise)), nlp(str(hypothesis))
    # Count how many negations we see between the 2 sentences - a mismatch indicates contradiction
    neg_p = sum(1 for t in p_doc if t.dep_ == "neg")
    neg_h = sum(1 for t in h_doc if t.dep_ == "neg")
    # Removes punctuation and stopwords
    p_set = {t.lemma_.lower() for t in p_doc if not t.is_stop and not t.is_punct}
    h_set = {t.lemma_.lower() for t in h_doc if not t.is_stop and not t.is_punct}
    # Determines the overlap between sentences using Jaccard Similarity
    overlap = len(p_set & h_set) / len(p_set | h_set) if (p_set | h_set) else 0.0
    return [float(neg_p), float(neg_h), float(abs(neg_p - neg_h)), float(overlap)]

# Factory to handle HuggingFace map function
def make_preprocess_fn(tokenizer):
    def preprocess(example):
        main_enc = tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=128)

        # Extract the relevant sentences for each expert
        p_nouns = get_pos_filtered_text(example["premise"], ["NOUN", "PROPN"])
        h_nouns = get_pos_filtered_text(example["hypothesis"], ["NOUN", "PROPN"])
        p_verbs = get_pos_filtered_text(example["premise"], ["VERB"])
        h_verbs = get_pos_filtered_text(example["hypothesis"], ["VERB"])

        # Encoders for the 2 experts
        ent_enc = tokenizer(p_nouns, h_nouns, truncation=True, padding="max_length", max_length=128)
        act_enc = tokenizer(p_verbs, h_verbs, truncation=True, padding="max_length", max_length=128)

        # Returns the sentences which each expert relies on
        return {
            "input_ids": main_enc["input_ids"],
            "attention_mask": main_enc["attention_mask"],

            "entity_input_ids": ent_enc["input_ids"],
            "entity_attention_mask": ent_enc["attention_mask"],

            "action_input_ids": act_enc["input_ids"],
            "action_attention_mask": act_enc["attention_mask"],

            "logic_features": [float(x) for x in extract_logic_features(example["premise"], example["hypothesis"])],

            "labels": int(example["label"])
        }
    return preprocess

## 4. Training and Saving
Train two separate models and then combine their predictions into an ensemble result. The weights are saved for later inference.

In [17]:
from transformers import EarlyStoppingCallback, DefaultDataCollator

optimal_configs = {
    "deberta": {
        "lr": 3e-5,
        "batch_size": 16,
        "epochs": 12
    },
    "modernbert": {
        "lr": 3e-5,
        "batch_size": 8,
        "epochs": 8
    }
}

backbones = {
    "deberta": "microsoft/deberta-v3-small",
    "modernbert": "answerdotai/ModernBERT-base"
}


In [18]:
# Output of the trained model
ensemble_save_path = "nli_ensemble_model"
os.makedirs(ensemble_save_path, exist_ok=True)

all_logits = {}

for name, path in backbones.items():
    # Load config and construct the base model
    config = optimal_configs[name]
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = POSSpecializedMoE(path).to(device)

    # Preprocess the data for training
    prep_fn = make_preprocess_fn(tokenizer)
    train_ds = Dataset.from_pandas(trainingDataframeWithGenerated).map(prep_fn)
    dev_ds = Dataset.from_pandas(devDataframe).map(prep_fn)

    args = TrainingArguments(
        output_dir=f"moe_{name}_optimized",
        learning_rate=config["lr"],
        per_device_train_batch_size=config["batch_size"],
        num_train_epochs=config["epochs"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True, # Keeps the best model seen during training
        metric_for_best_model="accuracy",
        save_total_limit=1,
        fp16=False, # GPU precision
        bf16=False, # GPU precision
        report_to="none"
    )

    def compute_metrics_fn(pred):
        predictions = pred.predictions
        labels = pred.label_ids

        preds = np.argmax(predictions, axis=1)
        acc = accuracy_score(labels, preds)

        return {
            "accuracy": acc
        }

    # Handles the training loop
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        data_collator=DefaultDataCollator(),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        compute_metrics=compute_metrics_fn
    )

    trainer.train()

    specific_save_dir = os.path.join(ensemble_save_path, name)
    os.makedirs(specific_save_dir, exist_ok=True)

    print(f"Saving {name} to {specific_save_dir}...")

    # Store the weights from the trained model
    torch.save(model.state_dict(), os.path.join(specific_save_dir, "moe_weights.pt"))

    tokenizer.save_pretrained(specific_save_dir)

    preds = trainer.predict(dev_ds)
    all_logits[name] = preds.predictions

final_logits = (all_logits["deberta"] + all_logits["modernbert"]) / 2
final_preds = np.argmax(final_logits, axis=1)

print("\n" + "="*50)
print("--- Final Optimised Ensemble Metrics ---")
print(f"Accuracy: {accuracy_score(devDataframe['label'], final_preds):.4f}")
print(f"Macro F1: {f1_score(devDataframe['label'], final_preds, average='macro'):.4f}")
print("="*50)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/25432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.341240,0.292241,0.877375
2,0.214554,0.416089,0.872922
3,0.124349,0.571668,0.878266
4,0.072124,0.769923,0.877375
5,0.061240,0.762152,0.883017
6,0.038706,0.840327,0.873515
7,0.033873,0.897470,0.887173
8,0.022579,0.921048,0.883314
9,0.014547,0.995101,0.888213
10,0.008855,0.997860,0.889549


...

Map:   0%|          | 0/25432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.360159,0.334911,0.865796
2,0.214944,0.494910,0.881977
3,0.095361,0.626661,0.884947
4,0.035434,0.876329,0.878266
5,0.025468,0.986885,0.881532
6,0.005932,1.071103,0.884056


Saving modernbert to nli_ensemble_model/modernbert...



--- Final Optimised Ensemble Metrics ---
Accuracy: 0.8953
Macro F1: 0.8952


### Evaluation of Model Training

In [19]:
from matplotlib import pyplot as plt

def plot_training_history(trainer):
    logs = trainer.state.log_history

    train_loss = []
    eval_loss = []

    for log in logs:
        if "loss" in log:
            train_loss.append(log["loss"])
        if "eval_loss" in log:
            eval_loss.append(log["eval_loss"])

    plt.figure()
    plt.plot(train_loss, label="Train Loss")
    if len(eval_loss) > 0:
        plt.plot(eval_loss, label="Eval Loss")

    plt.xlabel("Steps")
    plt.ylabel("Loss")
    plt.title("Training Curve")
    plt.legend()
    plt.show()


plot_training_history(trainer)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def analyse_gate_usage(model, dataset, num_samples=100):
    model.eval()
    gate_usage = []

    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]

        batch = {}
        for k, v in sample.items():
            if k == "labels":
                continue
            batch[k] = torch.tensor(v).unsqueeze(0)

        with torch.no_grad():
            out = model(**batch)

        if "gate_weights" in out:
            gate_usage.append(out["gate_weights"].cpu().numpy()[0])

    if len(gate_usage) == 0:
        print("No gate weights returned by model")
        return

    gate_usage = np.array(gate_usage)

    print("\n=== Average Gate Weights ===")
    print("Semantic | Entity | Action | Logic")
    print(gate_usage.mean(axis=0))


analyse_gate_usage(model, dev_ds)